<a href="https://colab.research.google.com/github/arlylelu25-debug/rtm1_2501010335/blob/main/notebooks/akuisisi_openstreetmap_bali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Akuisisi Data Wisata Bali - OpenStreetMap via Overpass API
Sumber: OpenStreetMap (ODbL) | Output: CSV di data/raw/

In [ ]:

import requests, time, pandas as pd
from pathlib import Path
from datetime import datetime

HEADERS = {"User-Agent": "BIFP-243-RTM1 (tugas kuliah INSTIKI)"}
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:60];
area["name"="Bali"]["admin_level"="4"]->.bali;
(
  node["tourism"="attraction"](area.bali);
  way["tourism"="attraction"](area.bali);
  relation["tourism"="attraction"](area.bali);
  node["tourism"="museum"](area.bali);
  way["tourism"="museum"](area.bali);
  node["tourism"="viewpoint"](area.bali);
  node["tourism"="theme_park"](area.bali);
  node["tourism"="zoo"](area.bali);
  node["tourism"="aquarium"](area.bali);
);
out center;
"""

response = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=60)
print("Status code:", response.status_code)
time.sleep(1)
data_json = response.json()
print("Jumlah dari API:", len(data_json.get("elements", [])))

rows = []
for el in data_json.get("elements", []):
    tags = el.get("tags", {})
    lat = el.get("lat", el.get("center", {}).get("lat"))
    lon = el.get("lon", el.get("center", {}).get("lon"))
    if lat is None or lon is None:
        continue
    rows.append({
        "nama_wisata": tags.get("name", ""),
        "kategori": tags.get("tourism", ""),
        "latitude": lat,
        "longitude": lon,
        "osm_id": el.get("id"),
        "osm_type": el.get("type"),
        "sumber": "OpenStreetMap"
    })

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["osm_id","osm_type"])
print("Jumlah record akhir:", len(df))
print(df.head())

tanggal = datetime.now().strftime("%Y%m%d")
Path("data/raw").mkdir(parents=True, exist_ok=True)
path_csv = f"data/raw/openstreetmap_bali_{tanggal}.csv"
df.to_csv(path_csv, index=False, encoding="utf-8")
print("CSV tersimpan:", path_csv)

Status code: 200
Jumlah dari API: 713
Jumlah record akhir: 713
                             nama_wisata    kategori  latitude   longitude  \
0              Small Beautiful Waterfall  attraction -8.175709  115.106360   
1  Sunrise viewpoint of Tanah Lot Temple   viewpoint -8.619558  115.086291   
2                               Bali Zoo         zoo -8.591266  115.264542   
3                    Bali Butterfly Park  attraction -8.479459  115.134629   
4                           Museum Subak      museum -8.544466  115.138400   

      osm_id osm_type         sumber  
0  275573942     node  OpenStreetMap  
1  597770793     node  OpenStreetMap  
2  603098166     node  OpenStreetMap  
3  641637893     node  OpenStreetMap  
4  838277764     node  OpenStreetMap  
CSV tersimpan: data/raw/openstreetmap_bali_20261115.csv


# Provenance & Hash
Bukti asal data, ukuran file, dan MD5 untuk verifikasi.

In [2]:

import hashlib, json
from pathlib import Path

path_csv = "data/raw/openstreetmap_bali_20261115.csv"
with open(path_csv, "rb") as f:
    md5_hash = hashlib.md5(f.read()).hexdigest()

provenance = {
    "nama_berkas": path_csv,
    "ukuran_byte": Path(path_csv).stat().st_size,
    "hash_md5": md5_hash,
    "sumber": "OpenStreetMap via Overpass API",
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "jumlah_record": 713,
    "data_pribadi": "Tidak diambil"
}

Path("docs").mkdir(parents=True, exist_ok=True)
with open("docs/provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2, ensure_ascii=False)

print("MD5:", md5_hash)
print("Ukuran:", Path(path_csv).stat().st_size, "byte")
print("provenance.json tersimpan")

MD5: 72d502e47e3fb293ead2d592b1e7e62d
Ukuran: 55268 byte
provenance.json tersimpan
